# Lecture 33 Worksheet: Multi-Head Attention and the Encoder Block
**NAVTTC / NIAI AI/ML Trainer Cohort** — runs in Google Colab or local Jupyter (PyTorch, CPU is enough). Fill every `None` / `# YOUR CODE` area and run top to bottom; check cells print `OK` or tell you what to fix.

Last time you built ONE head of self-attention. Today you build the rest of a Transformer encoder layer: several heads in parallel, layer normalisation, the feed-forward layer, residual connections, and finally the **encoder block**. Then you **measure** what happens when a deep stack of blocks loses its residuals or its layer norm. About 10 minutes of CPU time in total.

In [1]:
import os, math, json, time, random
os.environ["OMP_NUM_THREADS"] = "2"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
torch.set_num_threads(2); np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0); np.random.seed(0); random.seed(0)
R = {}          # results we collect, printed as one JSON line at the very end
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("torch", torch.__version__)

torch 2.11.0+cu130


## Provided: the code from Lecture 32
(You wrote this last time. We include it so this notebook runs on its own.)

In [2]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None: scores = scores.masked_fill(~mask, float("-inf"))
    w = F.softmax(scores, dim=-1)
    return w @ V, w

def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float(); div = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    pe[:, 0::2] = torch.sin(pos / div); pe[:, 1::2] = torch.cos(pos / div); return pe
print("ok")

ok


## Task 1: Split into heads, merge back

With `h` heads each head works on `d_k = d / h` columns. Write `split_heads(x, h)` that turns `(B, n, d)` into `(B, h, n, d_k)` and `merge_heads(x)` that turns it back.

Hint: `x.view(B, n, h, d_k).transpose(1, 2)` and the reverse with `.transpose(1, 2).reshape(B, n, h * d_k)`.

In [3]:
def split_heads(x, h):
    B, n, d = x.shape
    d_k = d // h
    return x.view(B, n, h, d_k).transpose(1, 2)

def merge_heads(x):
    B, h, n, dk = x.shape
    return x.transpose(1, 2).reshape(B, n, h * dk)

In [4]:
x = torch.arange(2 * 3 * 8, dtype=torch.float32).view(2, 3, 8)
sx = split_heads(x, 4)
check("split shape (2, 4, 3, 2)", tuple(sx.shape) == (2, 4, 3, 2))
check("head 0 holds columns 0-1 of every word", torch.equal(sx[:, 0], x[:, :, 0:2]) and torch.equal(sx[:, 3], x[:, :, 6:8]))
check("merge undoes split", torch.equal(merge_heads(sx), x))

OK   split shape (2, 4, 3, 2)
OK   head 0 holds columns 0-1 of every word
OK   merge undoes split


## Task 2: Multi-head attention

Fill in `forward`: project the inputs with `W_q`, `W_k`, `W_v`, split into heads, call `attention` (it works on the last two dimensions, so all heads run at once), merge, and apply `W_o`. Return `(output, weights)`.

Careful with the mask: `True` = keep, shape broadcastable to `(B, h, n_q, n_k)`.

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.d, self.h = d, h
        self.W_q = nn.Linear(d, d, bias=False); self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False); self.W_o = nn.Linear(d, d, bias=False)
    def forward(self, q_in, kv_in, mask=None):
        Q = split_heads(self.W_q(q_in), self.h)
        K = split_heads(self.W_k(kv_in), self.h)
        V = split_heads(self.W_v(kv_in), self.h)
        heads, w = attention(Q, K, V, mask)
        return self.W_o(merge_heads(heads)), w

In [6]:
torch.manual_seed(0)
mha = MultiHeadAttention(16, 4); x = torch.randn(2, 5, 16)
out, w = mha(x, x)
check("output (2, 5, 16)", tuple(out.shape) == (2, 5, 16))
check("weights (2, 4, 5, 5): one 5x5 table PER HEAD", tuple(w.shape) == (2, 4, 5, 5))
check("each row sums to 1", torch.allclose(w.sum(-1), torch.ones(2, 4, 5), atol=1e-5))
ref = nn.MultiheadAttention(16, 4, bias=False, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mha.W_q.weight, mha.W_k.weight, mha.W_v.weight], 0)); ref.out_proj.weight.copy_(mha.W_o.weight)
r_out, r_w = ref(x, x, x, need_weights=True, average_attn_weights=False)
check("matches PyTorch's own nn.MultiheadAttention (same weights)", torch.allclose(out, r_out, atol=1e-5) and torch.allclose(w, r_w, atol=1e-5), "check split/merge and the sqrt(d_k) (d_k = d / h)")
m = torch.tril(torch.ones(5, 5, dtype=torch.bool))[None, None]
check("causal mask works for every head", float(mha(x, x, m)[1].triu(1).abs().max()) == 0.0)

OK   output (2, 5, 16)
OK   weights (2, 4, 5, 5): one 5x5 table PER HEAD
OK   each row sums to 1
OK   matches PyTorch's own nn.MultiheadAttention (same weights)
OK   causal mask works for every head


/tmp/ipykernel_710/2151167531.py:13: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  check("causal mask works for every head", float(mha(x, x, m)[1].triu(1).abs().max()) == 0.0)


## Task 3: Do more heads cost more parameters?

For d = 64, count the parameters of `MultiHeadAttention(64, h)` for h = 1, 2, 4, 8, 16. Store the counts in the list `counts`.

In [7]:
counts = [sum(p.numel() for p in MultiHeadAttention(64, h).parameters()) for h in [1, 2, 4, 8, 16]]
heads_list = [1, 2, 4, 8, 16]

In [8]:
print(dict(zip(heads_list, counts)))
check("all equal 4 * d * d = 16384", all(c == 4 * 64 * 64 for c in counts))

{1: 16384, 2: 16384, 4: 16384, 8: 16384, 16: 16384}
OK   all equal 4 * d * d = 16384


## Task 4: Layer normalisation by hand

For each word vector separately (NOT across the batch): subtract its mean, divide by `sqrt(variance + eps)`, then scale and shift with `gamma` and `beta`. PyTorch uses the **biased** variance (divide by `d`).

In [9]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, keepdim=True, unbiased=False)
    return (x - mu) / torch.sqrt(var + eps) * gamma + beta

In [10]:
x = torch.randn(2, 5, 16) * 3 + 7
ln = nn.LayerNorm(16)
mine = layer_norm(x, ln.weight, ln.bias)
check("matches nn.LayerNorm", torch.allclose(mine, ln(x), atol=1e-5))
check("every word vector now has mean ~0", float(mine.mean(-1).abs().max()) < 1e-5)
check("... and std ~1", float((mine.std(-1, unbiased=False) - 1).abs().max()) < 1e-3)
print("before: mean %.2f std %.2f   after: mean %.4f std %.2f" % (x.mean(), x.std(), mine.mean(), mine.std()))

OK   matches nn.LayerNorm
OK   every word vector now has mean ~0
OK   ... and std ~1
before: mean 6.80 std 2.95   after: mean -0.0000 std 1.00


## Task 5: Feed-forward layer works on each word separately

Implement `FeedForward` as `Linear(d, d_ff) → ReLU → Linear(d_ff, d)`. Then test the claim from the notes: change ONLY word 2 of the sentence. Measure how much the output of word 0 changes (a) after the feed-forward layer and (b) after multi-head attention. Store the two numbers.

In [11]:
class FeedForward(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.l1 = nn.Linear(d, d_ff)
        self.l2 = nn.Linear(d_ff, d)
    def forward(self, x):
        return self.l2(F.relu(self.l1(x)))

torch.manual_seed(1)
ffn, mha2 = FeedForward(16, 64), MultiHeadAttention(16, 4)
x = torch.randn(1, 5, 16); x2 = x.clone(); x2[0, 2] += 1.0         # change ONLY word 2
change_ffn = float((ffn(x) - ffn(x2))[0, 0].abs().max())
change_attn = float((mha2(x, x)[0] - mha2(x2, x2)[0])[0, 0].abs().max())

In [12]:
print("word 0 changes by %.2e after the feed-forward layer, %.3f after attention" % (change_ffn, change_attn))
check("FFN is position-wise (word 0 unchanged)", change_ffn == 0.0)
check("attention mixes words (word 0 DID change)", change_attn > 1e-3)
check("FFN parameters = 2*d*d_ff + d_ff + d", sum(p.numel() for p in ffn.parameters()) == 2 * 16 * 64 + 64 + 16)

word 0 changes by 0.00e+00 after the feed-forward layer, 0.227 after attention
OK   FFN is position-wise (word 0 unchanged)
OK   attention mixes words (word 0 DID change)
OK   FFN parameters = 2*d*d_ff + d_ff + d


## Task 6: The encoder block

Post-norm, as in the original paper (Pages 8 and 10): `x = LayerNorm(x + Attention(x))`, then `x = LayerNorm(x + FeedForward(x))`. Add `Dropout` on each sub-layer's output. Mask: `True` = real word, `False` = padding (same idea as Lecture 31).

In [13]:
class EncoderBlock(nn.Module):
    def __init__(self, d, h, d_ff, p=0.1):
        super().__init__()
        self.attn, self.ffn = MultiHeadAttention(d, h), FeedForward(d, d_ff)
        self.ln1, self.ln2, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, x, mask=None):
        attn_out, _ = self.attn(x, x, mask)
        x = self.ln1(x + self.drop(attn_out))
        ffn_out = self.ffn(x)
        return self.ln2(x + self.drop(ffn_out))

In [14]:
blk = EncoderBlock(16, 4, 64); blk.eval()
x = torch.randn(2, 6, 16)
check("block keeps the shape (2, 6, 16)", tuple(blk(x).shape) == (2, 6, 16))
pad = torch.tensor([[1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1]], dtype=torch.bool)[:, None, None, :]
a, b = blk(x, pad), blk(x.clone(), pad)
x_alt = x.clone(); x_alt[0, 4:] = torch.randn(2, 16) * 5        # change only the PADDED positions of sentence 0
check("padded positions do not influence the real words", torch.allclose(blk(x, pad)[0, :4], blk(x_alt, pad)[0, :4], atol=1e-5))
big = EncoderBlock(512, 8, 2048)
n_block = sum(p.numel() for p in big.parameters())
print("paper-size block (d=512, h=8, d_ff=2048):", n_block, "parameters; 6 blocks:", 6 * n_block)
check("block parameter count = 3,150,336", n_block == 3150336)

OK   block keeps the shape (2, 6, 16)
OK   padded positions do not influence the real words
paper-size block (d=512, h=8, d_ff=2048): 3150336 parameters; 6 blocks: 18902016
OK   block parameter count = 3,150,336


## Task 7: Do we really need the residual and the layer norm?
**The task.** 10 tokens from a vocabulary of 6. For every position the label is **how many OTHER positions hold the same token** (0 to 9). This needs the words to look at each other.

**The model.** Embedding + sinusoidal positions → a stack of **4 encoder blocks** (d = 32, 4 heads, d_ff = 64) → a linear layer per position. We train four versions that differ only inside the block: with both, residual only, layer norm only, neither. 3 seeds each, 4000 training and 1000 validation sequences, 15 epochs (Adam 0.002).

In [15]:
L, V = 10, 6
def make_count_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(0, V, (n, L), generator=g)
    y = (x.unsqueeze(2) == x.unsqueeze(1)).sum(2) - 1          # how many OTHER positions have the same token
    return x, y
xs, ys = make_count_data(2, 0); print(xs[0].tolist(), "->", ys[0].tolist())

[2, 3, 5, 0, 1, 3, 1, 1, 1, 3] -> [0, 2, 0, 0, 3, 2, 3, 3, 3, 2]


### Task 7a: A block with switches

Fill in `forward` so that `res=True` uses residual connections and `ln=True` uses layer norm. When a switch is off, that operation is simply skipped.

In [16]:
class SwitchBlock(nn.Module):
    def __init__(self, d, h, d_ff, res, ln):
        super().__init__()
        self.attn, self.ffn, self.res = MultiHeadAttention(d, h), FeedForward(d, d_ff), res
        self.ln1 = nn.LayerNorm(d) if ln else nn.Identity()
        self.ln2 = nn.LayerNorm(d) if ln else nn.Identity()
    def forward(self, x):
        a = self.attn(x, x)[0]
        x = self.ln1(x + a) if self.res else self.ln1(a)
        f = self.ffn(x)
        x = self.ln2(x + f) if self.res else self.ln2(f)
        return x

In [17]:
torch.manual_seed(0); xx = torch.randn(2, 5, 16)
full, none_ = SwitchBlock(16, 4, 32, True, True), SwitchBlock(16, 4, 32, False, False)
check("shapes", tuple(full(xx).shape) == (2, 5, 16) and tuple(none_(xx).shape) == (2, 5, 16))
check("full block output is normalised (mean ~ 0)", float(full(xx).mean(-1).abs().max()) < 1e-4)
check("no-LN block output is NOT normalised", float(none_(xx).mean(-1).abs().max()) > 1e-3)
check("residual path: with res, zeroing attention and FFN returns the input",
      (lambda b: (b.attn.W_o.weight.data.zero_(), b.ffn.l2.weight.data.zero_(), b.ffn.l2.bias.data.zero_(), torch.allclose(SwitchBlock.forward(b, xx), xx, atol=1e-5))[-1])(SwitchBlock(16, 4, 32, True, False)))

OK   shapes
OK   full block output is normalised (mean ~ 0)
OK   no-LN block output is NOT normalised
OK   residual path: with res, zeroing attention and FFN returns the input


In [18]:
class CountEncoder(nn.Module):
    def __init__(self, n_layers, res, ln, d=32, h=4):
        super().__init__()
        self.emb = nn.Embedding(V, d); self.register_buffer("pe", sinusoidal_pe(L, d)); self.d = d
        self.blocks = nn.ModuleList([SwitchBlock(d, h, 64, res, ln) for _ in range(n_layers)]); self.out = nn.Linear(d, L)
    def forward(self, x):
        h = self.emb(x) * math.sqrt(self.d) + self.pe
        for b in self.blocks: h = b(h)
        return self.out(h)

def train_count(res, ln, seed, n_layers=4, epochs=15, lr=2e-3):
    torch.manual_seed(seed)
    xt, yt = make_count_data(4000, seed); xv, yv = make_count_data(1000, seed + 50)
    model = CountEncoder(n_layers, res, ln); opt = torch.optim.Adam(model.parameters(), lr)
    for ep in range(epochs):
        perm = torch.randperm(4000)
        for i in range(0, 4000, 64):
            b = perm[i:i + 64]
            loss = F.cross_entropy(model(xt[b]).reshape(-1, L), yt[b].reshape(-1)); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        tr = float(F.cross_entropy(model(xt).reshape(-1, L), yt.reshape(-1)))
        return float((model(xv).argmax(-1) == yv).float().mean()), tr, model
t0 = time.time(); abl = {}; kept = None
for name, (res, ln) in {"both": (True, True), "residual only": (True, False), "layer norm only": (False, True), "neither": (False, False)}.items():
    runs = [train_count(res, ln, s) for s in (0, 1, 2)]
    if name == "both": kept = runs[0][2]
    abl[name] = {"acc": [r[0] for r in runs], "train_loss": [r[1] for r in runs]}
    print(f"{name:16s} val accuracy per seed {[round(r[0], 3) for r in runs]}   final train loss {[round(r[1], 3) for r in runs]}")
print("took %.0f s" % (time.time() - t0)); R["abl"] = abl

both             val accuracy per seed [0.999, 0.991, 0.998]   final train loss [0.005, 0.039, 0.005]
residual only    val accuracy per seed [0.992, 0.925, 0.983]   final train loss [0.031, 0.196, 0.04]
layer norm only  val accuracy per seed [0.966, 0.473, 0.417]   final train loss [0.12, 1.141, 1.323]
neither          val accuracy per seed [0.346, 0.393, 0.435]   final train loss [1.426, 1.344, 1.197]
took 247 s


### Task 7b: Read the result

Chance for this task is hard to state (labels are not uniform); just compare. Set each variable from the numbers above.

In [19]:
best = "both"
worst = "neither"
removing_one_is_ok = False

In [20]:
mean = lambda a: sum(a) / len(a)
check("worst variant", worst == min(abl, key=lambda k: mean(abl[k]["acc"])))
check("best variant", best == max(abl, key=lambda k: mean(abl[k]["acc"])))
check("removing one is ok", removing_one_is_ok == (mean(abl["residual only"]["acc"]) > 0.9 and mean(abl["layer norm only"]["acc"]) > 0.9))

OK   worst variant
OK   best variant
OK   removing one is ok


### What do the four heads look at?
We take the trained `both` model (seed 0) and plot the attention weights of its **first block**, one panel per head, for one validation sequence. We just look: whatever pattern appears is what the heads learned on THIS task.

In [21]:
xv, yv = make_count_data(1, 777)
with torch.no_grad():
    h0 = kept.emb(xv) * math.sqrt(kept.d) + kept.pe
    _, w = kept.blocks[0].attn(h0, h0)          # (1, 4 heads, 10, 10)
fig, axs = plt.subplots(1, 4, figsize=(10, 2.9))
for hd in range(4):
    axs[hd].imshow(w[0, hd].numpy(), cmap="Blues", vmin=0, vmax=1)
    axs[hd].set_title(f"head {hd + 1}", fontsize=10); axs[hd].set_xticks(range(10)); axs[hd].set_yticks(range(10))
    axs[hd].set_xticklabels([str(int(t)) for t in xv[0]], fontsize=7); axs[hd].set_yticklabels([str(int(t)) for t in xv[0]], fontsize=7)
fig.suptitle("Block 1 attention, tokens: " + " ".join(str(int(t)) for t in xv[0]), fontsize=10); fig.tight_layout(); fig.savefig("fig_heads.png", dpi=170); plt.close(fig)
same = (xv[0].unsqueeze(0) == xv[0].unsqueeze(1)).float()
for hd in range(4):
    on_same = float((w[0, hd] * same).sum(-1).mean())          # average attention mass a word puts on words with the SAME token
    print(f"head {hd + 1}: average attention on same-token positions = {on_same:.2f}   (uniform attention would give {float(same.mean(-1).mean()):.2f})")
    R.setdefault("head_same", []).append(on_same)
R["uniform_same"] = float(same.mean(-1).mean())

head 1: average attention on same-token positions = 0.40   (uniform attention would give 0.32)
head 2: average attention on same-token positions = 0.00   (uniform attention would give 0.32)
head 3: average attention on same-token positions = 0.00   (uniform attention would give 0.32)
head 4: average attention on same-token positions = 0.00   (uniform attention would give 0.32)


## Wrap-up
- A head is a slice of the vector; `h` heads cost the same parameters as one (`4·d²`).
- Attention mixes words; the feed-forward layer works on each word alone.
- One encoder block = attention → add & norm → feed-forward → add & norm. A paper-size block (d = 512) has 3,150,336 parameters, two thirds of them in the feed-forward layer.
- Residual connections and layer norm are what make a deep stack trainable (here: both removed → it barely trains).
- Next lecture: the decoder (masked self-attention and cross-attention) and training a full Transformer on translation.

**Think about it:** the notes (Page 8) put layer norm AFTER each sub-layer. Many modern models put it BEFORE the sub-layer. What do you expect to change when the stack gets 100 blocks deep? Write down your guess. We do not test it here.

In [22]:
print("RESULT_JSON " + json.dumps(R))

RESULT_JSON {"abl": {"both": {"acc": [0.9987000226974487, 0.9908999800682068, 0.9976000189781189], "train_loss": [0.005334383342415094, 0.038828108459711075, 0.005319039337337017]}, "residual only": {"acc": [0.9919000267982483, 0.9247999787330627, 0.9828000068664551], "train_loss": [0.030557727441191673, 0.19575436413288116, 0.04047360643744469]}, "layer norm only": {"acc": [0.9657999873161316, 0.4733999967575073, 0.4169999957084656], "train_loss": [0.11962376534938812, 1.1408175230026245, 1.3230934143066406]}, "neither": {"acc": [0.34610000252723694, 0.39250001311302185, 0.4352000057697296], "train_loss": [1.4259880781173706, 1.3440157175064087, 1.1966747045516968]}}, "head_same": [0.4000000059604645, 2.8534175271488493e-06, 1.8151528891063862e-11, 4.39705909229815e-05], "uniform_same": 0.3199999928474426}
